<a href="https://colab.research.google.com/github/stfnnnnnnn/karl-mangahas-flyrank/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stfnnnnnnn/1st-act/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*



### Finding A — Finding #1: The Anatomy of Growing Content

>The paper compares pages with rising impressions against pages with falling impressions. It reports that growing content averaged **3,180 words versus 2,311 words** for declining content (37.6% longer) and **184 days versus 230 days** in age (20% younger). The comparison is large — about 74K rising pages versus 45K falling pages — and the paper appropriately describes the result as an **observational comparison**. Trend direction is defined elsewhere in the paper from the most recent 30-day impression change versus the previous 30 days.

**Methodology question: where does the label come from, and what is independent of it?** *How exactly was the growing-versus-declining label constructed for each page, and are all variables being compared measured independently of the period used to assign that label?*

>Because the label comes from recent 30-day versus prior-30-day impression movement, I would want the analysis to make the timeline explicit. I would also ask whether the age and word-count gaps remain within the same brand/client, or under a client-grouped analysis. That would help distinguish a portfolio-level association from differences in client mix, publishing strategy, or content population.

**Safe interpretation:** In this portfolio, growing and declining pages **showed measured differences** in age and depth. The result is useful as a directional signal for investigation, but it does not show that making a page longer or younger would cause growth.

### Finding B — Finding #4: The Freshness Multiplier

>The paper identifies **31–90 days since update** as the strongest stable freshness band, with a reported growth-to-decline ratio of **7.88:1**. It also reports that 365+ day content refreshed within 30 days showed a **3.2× health-score difference** (10.7 to 34.5) and roughly **57× higher impressions** (71 to 4,039). Importantly, the paper also warns against over-reading the 361+ freshness bucket because its 283:1 ratio is based on only one declining page.

**Methodology question — does the design support an intervention claim?** *Were mature pages that received a recent refresh comparable to mature pages that did not before the refresh occurred?*

>Refresh assignment is unlikely to be random: teams may preferentially refresh pages with stronger historical demand, strategic importance, or better prior visibility. I would therefore ask whether the comparison controls for pre-refresh performance, client, age, and prior visibility, and whether the outcome measurements occur strictly after the refresh. A matched within-client or time-aware pre/post design with comparable unrefreshed pages would better separate the refresh association from selection effects.

**Safe interpretation:**

>The paper **observed a large cohort difference** between recently refreshed and less-recently updated mature pages. That supports refresh status as a decision-support signal for review prioritization; by itself, the observational comparison does not establish that the refresh caused the measured lift.

### Why these are useful methodology questions

>The first asks **where the outcome label comes from and whether the comparison respects that construction**. The second asks **whether the validation/comparison design can carry an intervention-style interpretation**. Both follow the same standard I apply to my Week-5 model below: make the timeline explicit, test repeated entities honestly, inspect leakage, and keep the wording no stronger than the evidence.



In [ ]:
# Reference values transcribed from the supplied March 2026 FlyRank paper.
# These constants make the markdown summary above easy to audit; they are not model inputs.
paper_checks = {
    "finding_1_growing_avg_words": 3180,
    "finding_1_declining_avg_words": 2311,
    "finding_1_growing_avg_age_days": 184,
    "finding_1_declining_avg_age_days": 230,
    "finding_4_stable_freshness_growth_to_decline_ratio": 7.88,
    "finding_4_mature_refresh_health_before": 10.7,
    "finding_4_mature_refresh_health_after": 34.5,
    "finding_4_mature_refresh_health_difference_x": 3.2,
    "finding_4_mature_refresh_impressions_comparison": "71 vs 4039 (~57x)",
}
paper_checks

{'finding_1_growing_avg_words': 3180,
 'finding_1_declining_avg_words': 2311,
 'finding_1_growing_avg_age_days': 184,
 'finding_1_declining_avg_age_days': 230,
 'finding_4_stable_freshness_growth_to_decline_ratio': 7.88,
 'finding_4_mature_refresh_health_before': 10.7,
 'finding_4_mature_refresh_health_after': 34.5,
 'finding_4_mature_refresh_health_difference_x': 3.2,
 'finding_4_mature_refresh_impressions_comparison': '71 vs 4039 (~57x)'}

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### What changed from Week 5

>In Week 5 I already used `GroupShuffleSplit` by `client_hash_id`, which was a good step because it tested the model on clients it had not seen during training. I also used the leakage-safe historical feature set `imp_prev30`, `clk_prev30`, and `pos_prev30`, so the main issue entering this audit is no longer correcting the feature timeline.

>What stood out instead was the **difference between the model's overall discrimination and its performance at the top of the ranking**. On the Week 5 grouped holdout, Random Forest achieved a ROC-AUC of **0.533** and Average Precision of **0.283**, while Precision@20 reached **50%** and Precision@50 reached **48%** against a **26% decline base rate**. This suggests that the model may be more useful for prioritizing a limited review queue than for separating declining and non-declining pages across the full dataset, but one grouped split is not enough to know how stable that result is.

>For this audit, I therefore rerun the same leakage-safe Random Forest two ways so I can see how much the validation design affects the measured performance:

>1. **Before — random row split:** individual pages are split without respecting client membership, so pages from the same client can appear on both sides.
>2. **After — grouped client split:** whole clients are held out, matching the validation design used in Week 5.

>The random row split is useful as a comparison because it represents an easier validation setting. If pages from the same client appear in both training and testing, the model may benefit from client-specific patterns that would not be available when ranking pages for a completely unseen client. The grouped split asks the harder and more relevant question: *can the model rank pages for clients it has never seen during training?*

>After running both versions, I compare their **ROC-AUC, Average Precision, Precision@20, and Precision@50**, together with the test-set decline base rate. I also check the number of clients shared between training and testing. The random split is expected to contain overlapping clients, while the grouped split should contain **zero client overlap** by construction.

>The grouped result remains the one I will use for my main claims because it better matches the intended use of the model. The random result is retained only as an audit comparison. If the metrics change substantially between the two designs, that would show that validation choice has an important effect on the apparent model performance. If they remain similar, that would provide additional evidence that the Week 5 result is not simply explained by pages from the same clients appearing on both sides of the split.

In [2]:
%pip -q install duckdb huggingface_hub

import os, getpass, duckdb, numpy as np, pandas as pd
from IPython.display import display

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("HF token: ")

con = duckdb.connect()
con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"

# Same March development endpoint used in Week 5, but with position rebuilt in the prior window.
features = con.sql(f"""
WITH bounds AS (
    SELECT MAX(report_date) AS end_d
    FROM {FACT}
    WHERE month = '2026-03'
),
windowed AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,

        -- Later outcome-period impressions
        SUM(
            CASE
                WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                THEN f.gsc_impressions
                ELSE 0
            END
        ) AS imp_last30,

        -- Historical impressions
        SUM(
            CASE
                WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                THEN f.gsc_impressions
                ELSE 0
            END
        ) AS imp_prev30,

        -- Later outcome-period clicks
        SUM(
            CASE
                WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                THEN f.gsc_clicks
                ELSE 0
            END
        ) AS clk_last30,

        -- Historical clicks
        SUM(
            CASE
                WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                THEN f.gsc_clicks
                ELSE 0
            END
        ) AS clk_prev30,

        -- Historical average position
        AVG(
            CASE
                WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                THEN f.gsc_avg_position
            END
        ) AS pos_prev30,

        -- Later outcome-period position
        AVG(
            CASE
                WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                THEN f.gsc_avg_position
            END
        ) AS pos_last30

    FROM {FACT} f, bounds b

    WHERE f.report_date > b.end_d - INTERVAL 60 DAY
      AND f.report_date <= b.end_d

    GROUP BY
        f.client_hash_id,
        f.content_hash_id

    HAVING imp_prev30 >= 100
)

SELECT *
FROM windowed
""").df()

features["is_declining"] = (
    features["imp_last30"]
    < 0.8 * features["imp_prev30"]
).astype(int)

print(f"Rows: {len(features):,}")
print(
    f"Clients: "
    f"{features['client_hash_id'].nunique():,}"
)
print(
    f"Overall decline base rate: "
    f"{features['is_declining'].mean():.2%}"
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 82,025
Clients: 37
Overall decline base rate: 26.87%


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

SAFE_FEATURES = ["imp_prev30", "clk_prev30", "pos_prev30"]
X = features[SAFE_FEATURES].copy()
y = features["is_declining"].copy()
groups = features["client_hash_id"]

def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    k = min(k, len(scores))
    order = np.argsort(-scores)[:k]
    return float(y_true[order].mean())

def new_rf():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
        )),
    ])

def fit_eval(train_idx, test_idx, label):
    model = new_rf()
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    prob = model.predict_proba(X.iloc[test_idx])[:, 1]
    yt = y.iloc[test_idx].to_numpy()

    return model, prob, {
        "Validation": label,
        "Test rows": len(test_idx),
        "Test clients": groups.iloc[test_idx].nunique(),
        "Base rate": float(np.mean(yt)),
        "ROC-AUC": roc_auc_score(yt, prob),
        "Average Precision": average_precision_score(yt, prob),
        "Precision@20": precision_at_k(yt, prob, 20),
        "Precision@50": precision_at_k(yt, prob, 50),
    }

idx = np.arange(len(features))

# Audit comparison: ordinary row-random split.
tr_r, te_r = train_test_split(
    idx,
    test_size=0.25,
    random_state=42,
    stratify=y,
)
rf_random, prob_random, before = fit_eval(
    tr_r, te_r, "Random row split"
)

# Week 5 design: whole clients held out.
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42,
)
tr_g, te_g = next(gss.split(X, y, groups))
rf_grouped, prob_grouped, after = fit_eval(
    tr_g, te_g, "Client-grouped split"
)

split_comparison = pd.DataFrame([before, after])

display(split_comparison.style.format({
    "Base rate": "{:.2%}",
    "ROC-AUC": "{:.3f}",
    "Average Precision": "{:.3f}",
    "Precision@20": "{:.2%}",
    "Precision@50": "{:.2%}",
}))

random_overlap = len(
    set(groups.iloc[tr_r]) & set(groups.iloc[te_r])
)
grouped_overlap = len(
    set(groups.iloc[tr_g]) & set(groups.iloc[te_g])
)

print("Train/test client overlap — random:", random_overlap)
print("Train/test client overlap — grouped:", grouped_overlap)

assert grouped_overlap == 0

,Validation,Test rows,Test clients,Base rate,ROC-AUC,Average Precision,Precision@20,Precision@50
0,Random row split,20507,33,26.87%,0.561,0.306,20.00%,32.00%
1,Client-grouped split,26184,10,25.96%,0.531,0.282,35.00%,44.00%


Train/test client overlap — random: 33
Train/test client overlap — grouped: 0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Timeline

`previous 30 days (features)  →  decision point  →  most recent 30 days (label/outcome)`

### Week 5 feature audit

| Field | Role | Prediction-time status | Decision |
|---|---|---|---|
| `client_hash_id` | grouping key | known | split/group only; never a feature |
| `content_hash_id` | page key | known | joins/error inspection only; never a feature |
| `imp_prev30` | feature | prior window | keep |
| `clk_prev30` | feature | prior window | keep |
| `pos_prev30` | feature | prior window | keep |
| `imp_last30` | label ingredient | outcome window | exclude from X |
| `pos_last30` | outcome-period field | outcome window | exclude from X |
| `is_declining` | target | derived from later vs prior impressions | target only |

>The corrected Week 5 model already follows this timeline, so this audit does **not** discover a new leakage problem in its reported feature set. The final Week 5 inputs are all available before the later decline window.

>I still run an attack test using `pos_last30` deliberately. The purpose is to demonstrate how much the measured performance can change if I violate the time boundary and allow an outcome-period sibling metric into the model. I treat that version as intentionally invalid and never as a candidate model.

>I also verify that client and content IDs are not model features and that existing product scores or optimization flags are not used as predictive inputs.


In [4]:
# Leakage attack: deliberately replace historical position with outcome-period position.
# This model is intentionally invalid and is included only to test the feature boundary.

X_overlap = features[[
    "imp_prev30",
    "clk_prev30",
    "pos_last30",
]].copy()

overlap_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
    )),
])

overlap_model.fit(
    X_overlap.iloc[tr_g],
    y.iloc[tr_g],
)

prob_overlap = overlap_model.predict_proba(
    X_overlap.iloc[te_g]
)[:, 1]

yt = y.iloc[te_g].to_numpy()

leakage_comparison = pd.DataFrame([
    {
        "Feature set": "Intentionally invalid: pos_last30",
        "ROC-AUC": roc_auc_score(yt, prob_overlap),
        "Average Precision": average_precision_score(yt, prob_overlap),
        "Precision@20": precision_at_k(yt, prob_overlap, 20),
        "Precision@50": precision_at_k(yt, prob_overlap, 50),
    },
    {
        "Feature set": "Week 5 historical features: pos_prev30",
        "ROC-AUC": roc_auc_score(yt, prob_grouped),
        "Average Precision": average_precision_score(yt, prob_grouped),
        "Precision@20": precision_at_k(yt, prob_grouped, 20),
        "Precision@50": precision_at_k(yt, prob_grouped, 50),
    },
])

print(f"Grouped-test base rate: {yt.mean():.2%}")

display(leakage_comparison.style.format({
    "ROC-AUC": "{:.3f}",
    "Average Precision": "{:.3f}",
    "Precision@20": "{:.2%}",
    "Precision@50": "{:.2%}",
}))

rf_estimator = rf_grouped.named_steps["model"]
importance = pd.Series(
    rf_estimator.feature_importances_,
    index=SAFE_FEATURES,
).sort_values(ascending=False)

print("Historical-feature Random Forest importance:")
display(importance.to_frame("Importance"))


Grouped-test base rate: 25.96%


,Feature set,ROC-AUC,Average Precision,Precision@20,Precision@50
0,Intentionally invalid: pos_last30,0.531,0.294,100.00%,94.00%
1,Week 5 historical features: pos_prev30,0.531,0.282,35.00%,44.00%


Historical-feature Random Forest importance:


,Importance
pos_prev30,0.536279
imp_prev30,0.409325
clk_prev30,0.054396


### Real failure examples

>The aggregate metrics tell me whether the model has useful ranking signal, but they do not show me what kinds of pages the model gets wrong. I therefore inspect the mistakes from the **audited client-grouped Random Forest**, focusing on pages at the wrong ends of the ranking.

>A **high-scored non-declining page** is similar to a false positive for this review workflow: the model gives the page a high decline score even though it does not later meet my decline definition. If these pages appear near the top of the queue, they can consume limited review capacity that could have been spent on pages that actually decline.

>A **low-scored declining page** is similar to a false negative: the page actually meets my decline definition in the outcome window, but the model assigns it a relatively low score. For this use case, these errors matter because a genuinely declining page could be placed too far down the queue to receive early review.

>I am deliberately keeping these examples pseudonymized, and I do not try to invent a specific explanation for why an individual page was ranked incorrectly. With only historical impressions, clicks, and search position, there are several factors the model cannot observe — including seasonality, search-intent changes, competing pages, content changes, query mix, or client-specific events.

>What I want from this section is simpler: **do the highest-scored non-declining pages and lowest-scored declining pages reveal where this three-feature model is too limited?** The examples below help me interpret the strong Precision@20 and Precision@50 results from Week 5 alongside the model's much weaker overall discrimination. They are evidence about the limitations of the ranking, not proof of the cause of any individual error.

In [5]:
# Error examples from the audited grouped model.
err = features.iloc[te_g][[
    "client_hash_id", "content_hash_id", "imp_prev30", "clk_prev30", "pos_prev30", "is_declining"
]].copy()
err["prob_decline"] = prob_grouped
err["predicted"] = (err["prob_decline"] >= 0.5).astype(int)

false_pos = err[(err["is_declining"] == 0) & (err["predicted"] == 1)].sort_values("prob_decline", ascending=False)
false_neg = err[(err["is_declining"] == 1) & (err["predicted"] == 0)].sort_values("prob_decline", ascending=True)

print("False positives:", len(false_pos))
display(false_pos.head(5))
print("False negatives:", len(false_neg))
display(false_neg.head(5))

# Compact error-rate summary.
print("Audited grouped test rows:", len(err))
print("False-positive rate among actual negatives:",
      f"{len(false_pos) / max(1, (err['is_declining'] == 0).sum()):.2%}")
print("False-negative rate among actual positives:",
      f"{len(false_neg) / max(1, (err['is_declining'] == 1).sum()):.2%}")


False positives: 2831


,client_hash_id,content_hash_id,imp_prev30,clk_prev30,pos_prev30,is_declining,prob_decline,predicted
9675,client_fef1a8f436438636,content_0fc93224cf2ba5f0,165.0,0.0,47.680490,0,0.94,1
61428,client_62f4a7e64f5e0096,content_ed1b332b1cf85f2b,1256.0,0.0,0.294339,0,0.93,1
48711,client_62f4a7e64f5e0096,content_24d9a6edb273b304,116.0,0.0,38.669591,0,0.92,1
81905,client_65de48885f4ef01b,content_d3deb25effd3253f,100.0,0.0,5.603492,0,0.92,1
6473,client_62f4a7e64f5e0096,content_a7723b26e44dd54e,120.0,0.0,38.767005,0,0.92,1


False negatives: 5595


,client_hash_id,content_hash_id,imp_prev30,clk_prev30,pos_prev30,is_declining,prob_decline,predicted
62301,client_62f4a7e64f5e0096,content_0fcdf0deb75b03d2,100.0,0.0,7.522928,1,0.000000,0
47284,client_62f4a7e64f5e0096,content_1fd8d7606137dc44,10192.0,10.0,20.950843,1,0.000000,0
46486,client_62f4a7e64f5e0096,content_b432bad61855efb2,305.0,0.0,5.977823,1,0.003333,0
21831,client_62f4a7e64f5e0096,content_499ff4ad3829b26f,232.0,1.0,5.884966,1,0.003333,0
61947,client_62f4a7e64f5e0096,content_1fe8a8a4e913881f,178.0,0.0,5.306667,1,0.003333,0


Audited grouped test rows: 26184
False-positive rate among actual negatives: 14.60%
False-negative rate among actual positives: 82.30%


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### What I could overstate after Week 5

> “Random Forest achieved 50% Precision@20 and 48% Precision@50, so it reliably identifies pages that will decline.”

The measured top-K results are real for the Week 5 grouped holdout, but that sentence would go beyond the evidence. The same model has a ROC-AUC of only **0.533**, so its overall separation between declining and non-declining pages is weak. The strong top-of-queue result also comes from one grouped development split.

### How I would say it now

>On the March development slice, the leakage-safe Random Forest ranked the client-grouped holdout with **50% Precision@20 and 48% Precision@50**, compared with a **26% decline base rate**. Its ROC-AUC was **0.533** and Average Precision was **0.283**.

>These results suggest that the model concentrated more later-declining pages near the top of this particular review queue than the underlying base rate and the Week 4 baseline rule. However, the weak ROC-AUC means the model does not separate declining and non-declining pages strongly across the full holdout, and one grouped split is not enough to show that the top-K advantage will remain equally strong for other held-out client combinations.

>I therefore describe the Week 5 result as **promising directional ranking evidence that requires robustness testing**, not as a reliable automatic decline predictor. The appropriate use remains decision support: the score can help order pages for human review, but it does not determine why a page is at risk or what editorial action should be taken.

>I also cannot claim that editing a flagged page will improve its performance. This analysis evaluates whether historical search signals help prioritize later decline risk; it does not estimate the causal effect of a refresh or rewrite.

### What changed in my interpretation

- **Observed:** the reported Week 5 features are historical and precede the outcome window.
- **Measured:** the grouped holdout contains no client overlap and its base rate is reported beside the ranking metrics.
- **Directional:** the top of the queue is enriched relative to the base rate, but overall discrimination remains weak.
- **Decision-support:** the model can prioritize review candidates, but the ranking still requires further validation and human judgment.


In [6]:
# Final automated audit receipts.
assert "pos_last30" not in SAFE_FEATURES
assert "imp_last30" not in SAFE_FEATURES
assert "is_declining" not in SAFE_FEATURES
assert "client_hash_id" not in SAFE_FEATURES
assert "content_hash_id" not in SAFE_FEATURES
assert grouped_overlap == 0

print("PASS: Week 5 reported features are historical")
print("PASS: target/outcome-period fields excluded from SAFE_FEATURES")
print("PASS: IDs excluded from model features")
print("PASS: grouped train/test client overlap = 0")
print("PASS: audited feature set =", SAFE_FEATURES)

PASS: Week 5 reported features are historical
PASS: target/outcome-period fields excluded from SAFE_FEATURES
PASS: IDs excluded from model features
PASS: grouped train/test client overlap = 0
PASS: audited feature set = ['imp_prev30', 'clk_prev30', 'pos_prev30']


## Self-check

Before you submit, confirm each line honestly:

- [ - ] Every section above is filled — markdown thinking AND the code that backs it
- [ - ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ - ] No client names, URLs, or private queries anywhere
- [ - ] My claims use careful words: observed, measured, directional, decision-support
- [ - ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.